# 9 · Mobile manipulators

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rudip1/learn-manipulator-control/blob/main/2_notebooks/exercises/09_mobile_manipulators.ipynb)

Theory: [`1_theory/09_mobile_manipulators.md`](../../1_theory/09_mobile_manipulators.md) · notation:
[`00_notation.md`](../../1_theory/00_notation.md)

In [ ]:
# Install the module when it is not importable yet (Colab, fresh environment).
import importlib.util
if importlib.util.find_spec("manipulator_control") is None:
    %pip install -q git+https://github.com/Rudip1/learn-manipulator-control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import manipulator_control as mc

plt.rcParams.update({"figure.dpi": 80, "axes.grid": True})
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(9)

## Recap

- Unicycle base: $\dot x = v\cos\psi$, $\dot y = v\sin\psi$, $\dot\psi = \omega$ (eq. 9.1); no sideways motion
  (eq. 9.2, nonholonomic).
- One system: $T_e = T_b(\eta)\,T_m\,T_a(q_a)$ (eq. 9.3), configuration $q = (x, y, \psi, q_a)$, commands
  $\zeta = (\omega, v, \dot q_a)$ (eq. 9.4).
- Jacobian columns (eq. 9.5): $\omega \to (\hat z\times(p - p_b), \hat z)$; $v \to$ heading; arm $\to$ the arm's
  Jacobian rotated into the world.
- Exact arc integration of the base (eq. 9.6).
- Weights $W$ on $\zeta$ decide whether base or arm does the work; for a hierarchy substitute $\zeta = W^{-1/2}y$
  (eqs. 9.7–9.8).

## The robot

An anthropomorphic arm ($d_1 = 0.1$, $a_2 = 0.3$, $a_3 = 0.25$ m) mounted 0.1 m ahead of the base origin, 0.2 m up.
Frames: $-1$ end effector, $0$ base, $1..n+1$ the arm's DH frames. The tasks of chapters 7–8 work with this model
unchanged.

In [ ]:
mount = np.eye(4); mount[:3, 3] = [0.1, 0.0, 0.2]
robot = mc.MobileManipulator(mc.robots.anthropomorphic(), mount)
q0 = np.array([0.0, 0.0, 0.0, 0.0, 0.6, -1.2])
print("state size", robot.state_size(), " control size", robot.control_size())
print("end effector at start:", robot.frame_pose(q0, -1)[:3, 3])
print("Jacobian of the end effector (columns: omega, v, q1, q2, q3):\n", robot.frame_jacobian(q0, -1))

def draw_top(ax, q, color, alpha=1.0):
    x, y, psi = q[:3]
    ax.add_patch(plt.Circle((x, y), 0.17, color=color, alpha=0.25 * alpha))
    ax.plot([x, x + 0.17 * np.cos(psi)], [y, y + 0.17 * np.sin(psi)], color=color, linewidth=2, alpha=alpha)
    pts = np.array([robot.frame_pose(q, k)[:2, 3] for k in range(1, robot.arm_dof + 2)] + [robot.frame_pose(q, -1)[:2, 3]])
    ax.plot(pts[:, 0], pts[:, 1], "o-", color=color, linewidth=2, markersize=3, alpha=alpha)

## Reaching beyond the arm

The worked example of section 9.5: the target is 2.2 m away, the arm reaches 0.55 m. Three weightings; the third
adds the elbow limit of chapter 8 above the end-effector task.

In [ ]:
target = np.eye(4); target[:3, 3] = [2.0, 1.0, 0.3]

def reach(weights, elbow_limit=False, T=target, steps=1500):
    ee = mc.FrameTask("ee", -1, [0, 1, 2], T)
    elbow = mc.JointLimitTask("elbow", 4, -2.5, -0.5, 0.05, 0.15); elbow.set_gain(0.5)
    cfg = mc.PriorityConfig(inverse=mc.InverseMethod.DampedLeastSquares, damping=0.05, weights=np.asarray(weights, float))
    return mc.simulate_task_priority(robot, q0, [elbow, ee] if elbow_limit else [ee], cfg, 0.01, steps)

def arm_w(q):
    J = robot.frame_jacobian(q, -1)[:3, 2:]
    return np.sqrt(max(np.linalg.det(J @ J.T), 0.0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for k, (name, w, lim) in enumerate([("equal", [1] * 5, False), ("base 10", [10, 10, 1, 1, 1], False), ("base 10 + elbow limit", [10, 10, 1, 1, 1], True)]):
    r = reach(w, lim)
    ax[0].plot(r.q[:, 0], r.q[:, 1], color=f"C{k}", label=name); draw_top(ax[0], r.q[-1], f"C{k}")
    ax[1].semilogy(r.t, [arm_w(q) for q in r.q], color=f"C{k}", label=name)
    path = np.sum(np.linalg.norm(np.diff(r.q[:, :2], axis=0), axis=1))
    print(f"{name:22s} error {r.error_norm[-1, -1]:.1e} m, base drove {path:.2f} m, final elbow {r.q[-1, 5]:+.3f} rad")
draw_top(ax[0], q0, "0.5"); ax[0].plot(2.0, 1.0, "r*", markersize=13)
ax[0].set_aspect("equal"); ax[0].legend(fontsize=8); ax[1].set_title("arm manipulability"); ax[1].legend(fontsize=8)
plt.show()

## ✏️ Exercises

### ✏️ 1. The exact arc

Implement eq. (9.6), with the straight segment when $|\omega\Delta t|$ is tiny.

In [ ]:
def unicycle_step(eta, omega, v, dt):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
for _ in range(50):
    eta = rng.uniform(-2, 2, 3); omega, v, dt = rng.uniform(-2, 2), rng.uniform(-1, 1), rng.uniform(0.01, 1)
    assert np.allclose(unicycle_step(eta, omega, v, dt), mc.integrate_unicycle(eta, omega, v, dt))
assert np.allclose(unicycle_step(np.zeros(3), 0.0, 1.0, 2.0), [2.0, 0.0, 0.0])
print("✓ unicycle_step")

### ✏️ 2. The Jacobian, column by column

Build the $6\times5$ end-effector Jacobian of eq. (9.5) from `mc.geometric_jacobian(robot.arm, q_a)`, the base
pose and the mount.

In [ ]:
def mm_jacobian(q):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
for _ in range(20):
    q = rng.uniform(-np.pi, np.pi, 6)
    assert np.allclose(mm_jacobian(q), robot.frame_jacobian(q, -1), atol=1e-12)
print("✓ mm_jacobian")

### ✏️ 3. Weighted command by substitution

For one task, compute $\zeta$ with eq. (9.8): $D = W^{-1/2}$, $y = (JD)^\dagger v$, $\zeta = Dy$. Compare with the
library (`PriorityConfig(weights=...)`) and with the weighted pseudoinverse of chapter 6.

In [ ]:
def weighted_command(J, v, w):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
w = np.array([10.0, 10.0, 1.0, 2.0, 1.0])
for _ in range(10):
    q = rng.uniform(-np.pi, np.pi, 6); J = robot.frame_jacobian(q, -1)[:3]; v = rng.normal(size=3)
    ref = mc.task_priority_velocity([mc.PriorityLevel(J, v)], 5, mc.PriorityConfig(weights=w))
    assert np.allclose(weighted_command(J, v, w), ref)
    assert np.allclose(ref, mc.weighted_pseudoinverse(J, np.diag(w)) @ v)
print("✓ weighted_command")

## 🔨 Break it

### 🔨 1. Pretend the base is holonomic

Write the Jacobian as if the base had three free inputs $(\dot x, \dot y, \dot\psi)$, solve, and send the robot what
it can actually do: $v$ = the component of $(\dot x, \dot y)$ along the heading, $\omega = \dot\psi$. The sideways part
of every command is lost. Convergence slows down and the path differs from the plan.

In [ ]:
def holonomic_controller(q, T, k=1.0):
    p = robot.frame_pose(q, -1)[:3, 3]
    J = np.zeros((3, 6))
    J[:, 0] = [1, 0, 0]; J[:, 1] = [0, 1, 0]
    J[:, 2] = np.cross([0, 0, 1], p - np.array([q[0], q[1], 0.0]))
    J[:, 3:] = robot.frame_jacobian(q, -1)[:3, 2:]
    qdot = np.linalg.pinv(J) @ (k * (T[:3, 3] - p))
    psi = q[2]
    return np.concatenate([[qdot[2], qdot[0] * np.cos(psi) + qdot[1] * np.sin(psi)], qdot[3:]])

q, err = q0.copy(), []
for _ in range(1500):
    q = robot.integrate(q, holonomic_controller(q, target), 0.01)
    err.append(np.linalg.norm(target[:3, 3] - robot.frame_pose(q, -1)[:3, 3]))
right = reach([1] * 5)
plt.figure(figsize=(6, 3.2))
plt.semilogy(right.t, right.error_norm[:, 0], label="eq. (9.5): inputs (ω, v)")
plt.semilogy(right.t[1:], err, label="holonomic pretence")
plt.xlabel("t [s]"); plt.ylabel("end-effector error [m]"); plt.legend(); plt.show()
print(f"error after 15 s: correct {right.error_norm[-1, 0]:.1e} m, holonomic pretence {err[-1]:.1e} m")

### 🔨 2. Forget the lever arm of $\omega$

Drop $\hat z\times(p - p_b)$ from the $\omega$ column. For a position task that column is *only* the lever arm, so
it becomes zero: the controller believes turning the base does nothing, never turns it, and tries to reach a
target 1 m to the side by driving straight and stretching the arm.

In [ ]:
def no_lever_controller(q, T, k=1.0):
    J = robot.frame_jacobian(q, -1)[:3].copy()
    J[:, 0] = 0.0
    return np.linalg.pinv(J) @ (k * (T[:3, 3] - robot.frame_pose(q, -1)[:3, 3]))

q, err2 = q0.copy(), []
for _ in range(1500):
    q = robot.integrate(q, no_lever_controller(q, target), 0.01)
    err2.append(np.linalg.norm(target[:3, 3] - robot.frame_pose(q, -1)[:3, 3]))
print(f"error after 15 s without the lever arm: {err2[-1]:.1e} m;  base heading turned by {q[2]:.3f} rad")

### 🔨 3. Euler instead of the exact arc

Drive a full circle of radius 0.5 m ($v = 0.5$ m/s, $\omega = 1$ rad/s) with Euler steps and compare with the
exact arcs (9.6) step by step. (Over the *full* circle the Euler polygon happens to close — its steps are evenly
spaced directions — so look at the largest gap along the way.) The gap halves with $\Delta t$: Euler is first
order.

In [ ]:
for dt in [0.1, 0.05, 0.01]:
    eta_e, eta_x, gap = np.zeros(3), np.zeros(3), 0.0
    steps = int(round(2 * np.pi / dt)); dt_eff = 2 * np.pi / steps
    for _ in range(steps):
        eta_e = eta_e + dt_eff * np.array([0.5 * np.cos(eta_e[2]), 0.5 * np.sin(eta_e[2]), 1.0])
        eta_x = mc.integrate_unicycle(eta_x, 1.0, 0.5, dt_eff)
        gap = max(gap, np.linalg.norm(eta_e[:2] - eta_x[:2]))
    print(f"dt = {dt:5.2f}: largest gap between Euler and the exact arc {gap * 100:.2f} cm")

## What to remember

- Write base and arm as one system with the *commanded* velocities $(\omega, v, \dot q_a)$; the nonholonomic
  constraint is then built into the Jacobian.
- The $\omega$ column has a lever arm: turning the base swings the arm.
- Weights choose who works when there is a choice; a target out of the arm's reach needs the base anyway.
- A heavy base pushes the arm to its workspace boundary — protect the arm with limits or a manipulability task.
- Integrate the base exactly (arcs) in simulation; Euler drifts.